In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('users_7_7_2026 2_49_38 AM.csv')
print(df.sample)
print(df.info())

## Câu 1
print("Cau 1")
df.loc[df["Usage location"].notna(), "Usage location"] = "VN"

## Câu 2
print("Cau 2")
missing = df.isnull().sum() # Kiểm tra số lượng giá trị thiếu
print(missing[missing > 0].sort_values(ascending=False)) # Chỉ hiển thị các cột có NaN

## Xóa cột
drop_cols = [
    "Soft deletion time stamp",
    "AgeGroup",
    "Preferred data location",
    "Last dirsync time",
    "Fax"
]
df.drop(columns=drop_cols, inplace=True)
## Những cột này không có dữ liệu (1900 cái đều là NaN) nên xóa bỏ luôn 

## Điền "unknown"
fill_unknown = [
    "City",
    "CountryOrRegion",
    "StateOrProvince",
    "Preferred language"
]
for col in fill_unknown:
    df[col] = df[col].fillna("Unknown")
## Đối với các cột này, việc điền "Unknown" không tạo ra dữ liệu giả mà chỉ thể hiện rằng hiện chưa có thông tin. Điều này giúp dễ dàng thống kê, lọc và hiển thị dữ liệu.

## Những chỗ NaN còn lại sẽ được giữ nguyên vì đây là các thông tin cá nhân hoặc thông tin nghiệp vụ. Không có căn cứ để suy luận hoặc tự điền giá trị. Nếu tự ý điền sẽ tạo ra dữ liệu sai và ảnh hưởng đến chất lượng dữ liệu. 

## Câu 3
print("Cau 3")
# Kiểm tra kiểu dữ liệu trước khi chuyển
print(df[["When created", "Last password change time stamp"]].dtypes)

# Chuyển sang kiểu datetime
df["When created"] = pd.to_datetime(
    df["When created"],
    utc=True,
    errors="coerce"
)

df["Last password change time stamp"] = pd.to_datetime(
    df["Last password change time stamp"],
    utc=True,
    errors="coerce"
)

# Kiểm tra lại
print(df[["When created", "Last password change time stamp"]].dtypes)

## Câu 4
print("Cau 4")
# Hàm chuẩn hóa số điện thoại
def clean_phone(phone):
    if pd.isna(phone):
        return phone

    phone = str(phone).strip()                 # bỏ khoảng trắng đầu/cuối
    phone = phone.replace("'", "")             # bỏ dấu '
    phone = re.sub(r"\s+", "", phone)          # bỏ mọi khoảng trắng
    phone = phone.replace("-", "")             # bỏ dấu gạch ngang

    return phone

# Chuẩn hóa
df["Mobile Phone"] = df["Mobile Phone"].apply(clean_phone)
df["Phone number"] = df["Phone number"].apply(clean_phone)

# Xử lý dữ liệu nhập sai trong Preferred language
phone_pattern = r"^(\+84|0)\d{9}$"

mask = df["Preferred language"].fillna("").str.match(phone_pattern)

df.loc[mask, "Preferred language"] = "Invalid"

## Câu 5
print("Cau 5")
# Hiển thị các bản ghi có User principal name bị trùng
duplicate_users = df[df.duplicated(subset="User principal name", keep=False)]

print(duplicate_users)
print("Số bản ghi trùng:", df.duplicated(subset="User principal name").sum())
# Loại bỏ bản ghi trùng 
df = df.drop_duplicates(
    subset="User principal name",
    keep="first"
)
# Kiểm tra lại 
print("Số bản ghi trùng sau khi xử lý:",df.duplicated(subset="User principal name").sum())

<bound method NDFrame.sample of                            Display name  DirSyncEnabled  \
0                            Admin BKVN           False   
1                       Admin TDI Group           False   
2          ALPHA Procurement Department           False   
3           Bá Thị Trang (TDICONS-HCNS)           False   
4     Bạch Đào Sơn Thương (TDI MEP-PQS)           False   
...                                 ...             ...   
1895     Vương Trọng Hiếu (TDICONS-PTC)           False   
1896       Vương Văn Dũng (TDICONS-PTC)           False   
1897          Vương Văn Tùng (VBIM-PTK)           False   
1898                   Xe hơi 29H-18912           False   
1899         Yên Văn Hanh (TDI MEP-PQS)           False   

                                User principal name  \
0                            adminbkvn@bkvietnam.vn   
1                                   admin@tdimep.vn   
2                         procurement@vattualpha.vn   
3                                trangb